# Evaluasi 7 metode sentimen – Banyu Bening Resto (Capstone ASAH 2026)
**Versi 2 (2.000 review).** Jalankan dari atas ke bawah (Runtime > Run all).

**Persiapan:** unggah `reviews_v2.csv` (2.000 ulasan) ke panel Files di Colab sebelum menjalankan sel 4. Tanpa file itu, notebook tetap jalan tetapi hanya menguji skema A (latih sintetis).

**Isi:** 7 metode × 2 skema latih (A: sintetis saja; B: sintetis + review berlabel rating), ujian utama pada 100 komentar tulisan tangan, cek in-domain 5-fold, prediksi out-of-fold untuk 2.000 review, dan penyimpanan model.

**Catatan:** data sintetis dan test set ditulis manual, jadi belum membuktikan akurasi pada ulasan asli.

In [ ]:
import sklearn; print(sklearn.__version__)  # dilatih dengan 1.9.1; versi lain tetap bisa dijalankan, angka bisa sedikit berbeda

## 1. Bank frasa (pembangkit data latih sintetis)

In [ ]:
import random,re
# ---------- bank frasa (kata kunci: {i}=acuan item, {m}=menit tunggu) ----------
B={}
def b(asp,pol,*xs): B[(asp,pol)]=list(xs)
b('rasa','+',"{i} enak banget","{i} enak, bumbunya meresap","rasa {i} mantap, pas di lidah","{i} gurih dan nggak bikin eneg",
  "{i} juara sih","suka banget sama {i}","bumbu {i} pas, nggak kemanisan","{i} nggak ngecewain","rasanya otentik kayak masakan rumah",
  "{i} mantul","pedasnya pas","{i} segar dan wangi","{i} lezat, bakal pesan lagi","{i} bikin nambah terus","rasa {i} konsisten enak",
  "sambalnya nendang, cocok banget sama {i}","{i} matang sempurna","{i} dimasak dengan pas")
b('rasa','-',"{i} hambar","{i} kurang bumbu, rasanya datar","{i} keasinan","{i} terlalu manis buat saya","rasa {i} biasa aja, nggak sesuai ekspektasi",
  "{i} agak amis","{i} kurang matang","{i} agak gosong","bumbu {i} nggak meresap","{i} berminyak banget","{i} pedasnya kebangetan",
  "rasa {i} aneh, nggak cocok di lidah","{i} nggak seenak yang dibilang orang","{i} nggak enak sama sekali","{i} rasanya jauh dari harapan",
  "{i} kurang sesuai selera saya","{i} cuma enak di awal, habis itu eneg")
b('porsi','+',"porsinya banyak","porsinya pas buat kenyang","porsinya royal untuk harga segitu","kenyang banget","porsinya gede","isiannya melimpah","lauknya banyak")
b('porsi','-',"porsinya kecil","porsinya kurang buat saya","porsinya sedikit banget","isiannya pelit","lauknya sedikit","belum kenyang padahal sudah habis satu porsi penuh","porsinya nggak sebanding sama harga")
b('kecepatan','+',"pesanan datang cepat","nggak pakai lama, langsung diantar","disajikan cepat banget","waktu tunggunya singkat","nggak sampai {m} menit sudah datang","pesanan keluar dengan cepat meski lagi ramai","cepat banget keluarnya")
b('kecepatan','-',"nunggu {m} menit baru datang","lama banget, sampai {m} menit","pesanan datangnya lambat padahal cuma sedikit menu","sudah nunggu hampir {m} menit belum juga keluar",
  "waktu tunggunya kelamaan, sekitar {m} menit","pesanannya lama banget keluarnya","nunggunya nggak masuk akal, {m} menit","pesanan datang telat banget")
b('kecepatan','~',"agak lama, sekitar {m} menit","nunggunya lumayan lama, ±{m} menit","datangnya agak lambat","waktu tunggunya agak panjang")
b('suhu','+',"datang masih panas","masih sizzling waktu disajikan","hangat pas diantar","minumannya dingin dan seger","kuahnya panas mengepul","disajikan masih hangat")
b('suhu','-',"makanannya sudah dingin pas datang","kuahnya kurang panas","nasinya dingin","udah nggak panas waktu sampai di meja","esnya cuma sedikit jadi minuman hangat","makanan datang suam-suam kuku")
b('pelayanan','+',"pelayanannya ramah","stafnya sigap dan sopan","kasirnya ramah","pelayannya helpful","minta tambahan sambal langsung dikasih","mbak-mbaknya ramah banget","stafnya responsif")
b('pelayanan','-',"pelayannya kurang ramah","stafnya cuek","minta sendok dari tadi nggak dibawain","pelayanan kurang sigap","pas bertanya malah dicuekin","pesanan sempat salah bawa","ada pesanan yang kelewat")
b('harga','+',"harganya masuk akal","worth it untuk harga segini","harga bersahabat","murah untuk rasa segini","harganya terjangkau")
b('harga','-',"harganya agak mahal untuk porsi segitu","overpriced menurut saya","harganya kemahalan","nggak worth it sama harganya")
b('kebersihan','+',"tempatnya bersih dan nyaman","mejanya bersih","suasananya adem","peralatan makannya bersih","tempatnya rapi")
b('kebersihan','-',"meja masih kotor waktu kami duduk","tempatnya panas dan agak bau asap","lantainya licin dan kurang bersih","banyak lalat di sekitar meja","peralatan makannya kurang bersih")
b('qr','+',"pesan lewat QR di meja gampang banget","scan barcode, pilih menu, praktis","nggak perlu antre di kasir, enak","menu di QR jelas ada fotonya","bayar QRIS-nya lancar")
b('qr','-',"scan QR-nya agak ribet","menu di QR loadingnya lama","sudah bayar tapi status pesanan nggak berubah","pembayaran QRIS sempat gagal","QR-nya error jadi harus tanya kasir lagi")
b('kemasan','+',"dibungkus rapi, kuahnya nggak tumpah","kemasannya aman","bungkusnya rapi dan kokoh")
b('kemasan','-',"kemasannya bocor","kuah tumpah di tas","bungkusnya seadanya")
b('umum','+',"overall puas","pengalaman makan yang menyenangkan","recommended","pasti balik lagi","keluarga senang semua","nggak nyesel makan di sini","puas banget")
b('umum','-',"kecewa sama pengalaman makan kali ini","kurang puas secara keseluruhan","nggak sesuai ekspektasi","mungkin lagi apes, tapi kurang oke","pengalaman yang mengecewakan","nggak sesuai harapan sama sekali")
b('biasa','0',"biasa aja sih","standar, nggak ada yang istimewa","lumayan lah","cukup, belum bikin jatuh cinta","oke tapi masih bisa lebih baik","nggak jelek, nggak istimewa juga","rasanya sedang-sedang saja","ya begitu-begitu saja")
# frasa umum tambahan khusus korpus latih (tidak dipakai untuk komentar order)
GEN_POS=["bagus banget","sangat memuaskan","top deh","mantap jiwa","keren","the best","sesuai ekspektasi","lebih dari yang dibayangkan","layak dicoba","nggak kecewa sama sekali","puas dengan hasilnya","terbaik di kelasnya","bikin ketagihan","senang banget","pelayanan memuaskan","recommended banget","gokil enaknya","nikmat","worth the hype","nggak ada komplain"]
GEN_NEG=["mengecewakan banget","sangat mengecewakan","parah","zonk","nyesel datang","nggak akan balik lagi","tidak sesuai ekspektasi","jauh dari harapan","buruk sekali","bikin kesel","ilfeel","tidak layak untuk harganya","kapok","pengalaman terburuk","bikin emosi","tidak direkomendasikan","bad experience","nggak banget","tidak puas","disappointing"]
OPEN_POS_NEG_CONN={('+','+'):[", ",", "," dan ",". "],('-','-'):[", ",", "," dan ",". "],
 ('+','-'):[", tapi ",", cuma ",", sayangnya ",". Tapi "],('-','+'):[", tapi ",", untungnya ",". Tapi "],
 ('+','~'):[", tapi ",", cuma "],('~','+'):[", tapi "],('0','+'):[", tapi "],('0','-'):[", tapi "],('+','0'):[", "],('~','-'):[", "],('-','~'):[", "]}
CLOSE_POS=["Pasti balik lagi.","Recommended!","Bakal ajak teman kapan-kapan.","Lanjutkan ya!","Terima kasih."]
CLOSE_NEG=["Semoga ke depannya diperbaiki.","Mungkin nggak balik lagi.","Tolong dievaluasi ya.","Sayang sekali.","Semoga jadi masukan."]
SHORT_POS=["Mantap!","Juara","Recommended banget","enak","Puas!","Top"]
SHORT_NEG=["Kecewa","Parah, nggak lagi-lagi","Nggak enak","Mengecewakan","Kurang puas"]

COLLOQ={'Hotplate':['hotplate-nya'],'Sapo Tahu':['sapo tahunya'],'Mie':['mienya'],'Kwetiau':['kwetiaunya'],'Bihun':['bihunnya'],
 'Nasi Goreng':['nasi gorengnya'],'Aneka Nasi':['nasinya'],'Paket Personal':['paketnya'],'Paket Ber 5':['paketnya'],'Juice':['jusnya'],
 'Minuman Standar':['minumannya'],'Minuman Teko':['minumannya'],'Minuman Jumbo':['minumannya'],'Soda':['sodanya'],'Snack':['camilannya'],
 'Sayuran':['sayurnya'],'Soto':['sotonya'],'Gurame':['ikannya'],'Nila':['ikannya'],'Lele':['ikannya'],'Alacarte':[]}
MAIN_SKIP={'Sambal','Saus','Kemasan Take Away','FREE','Makanan','Minuman Standar','Minuman Teko','Minuman Jumbo','Juice','Soda'}
def clean_name(n): return n.replace(' / ons','')
def item_ref(name,cat,rng):
    nm=clean_name(name)
    opts=COLLOQ.get(cat,[])
    return rng.choice(opts+[nm,nm]) if opts else nm
def cap(s): return s[:1].upper()+s[1:] if s else s
FORMAL=[('nggak','tidak'),('banget','sekali'),(' aja',' saja'),('udah','sudah'),('kayak','seperti'),('cuma','hanya'),('nunggu','menunggu'),('nunggunya','menunggunya'),('pakai lama','lama')]
def formalize(s):
    for a,c in FORMAL: s=re.sub(r'(?<![A-Za-z])'+re.escape(a.strip())+r'(?![A-Za-z])',lambda m,c=c:c.strip() if m.group(0).islower() else c.strip().capitalize(),s)
    return s
def very_casual(s,rng):
    s=s[:1].lower()+s[1:]
    s=re.sub(r'nggak',lambda m: rng.choice(['gak','ga','nggak']),s)
    if rng.random()<.5: s=s.replace('banget','bgt')
    s=s.rstrip('.')
    for w in ['enak','mantap','lama','banget']:
        if w in s and rng.random()<.25:
            s=s.replace(w,w+w[-1]*rng.choice([1,2]),1); break
    return s
def compose(story,ctx,rng,short_ok=True,extra_gen=False):
    """story: list of (aspek,pol). ctx: i_main,i_full,m,when,dow,table,service,source,n_items,rush"""
    pol_main=story[0][1]
    # komentar sangat pendek
    if short_ok and rng.random()<0.06 and len(story)==1 and story[0][0]!='kecepatan':
        txt=rng.choice(SHORT_POS if pol_main=='+' else SHORT_NEG if pol_main=='-' else [rng.choice(B[('biasa','0')])])
        return finish(txt,rng)
    parts=[]; pols=[]
    for asp,pol in story:
        ph=rng.choice(B[(asp,pol)])
        ph=ph.format(i=ctx['i_main'],m=ctx['m'])
        parts.append(ph); pols.append(pol)
    if extra_gen and rng.random()<.5:
        g=rng.choice(GEN_POS if pol_main=='+' else GEN_NEG if pol_main=='-' else [rng.choice(B[('biasa','0')])])
        parts.append(g); pols.append(pol_main)
    body=parts[0]
    for k in range(1,len(parts)):
        conn=rng.choice(OPEN_POS_NEG_CONN.get((pols[k-1],pols[k]),[", "]))
        body+=conn+(cap(parts[k]) if conn.startswith('.') else parts[k]) if True else ''
    body=cap(body)
    if not body.endswith('.'): body+='.'
    # pembuka
    op=None
    if ctx.get('opener',True) and rng.random()<0.5:
        cands=[]
        if ctx['service']=='dine_in':
            cands+=[f"Makan {ctx['when']} di sini hari {ctx['dow']}",f"Mampir pas {ctx['dow']} {ctx['when']}",f"Datang {ctx['when']} hari {ctx['dow']}",f"Duduk di meja {ctx['table']}",
                    f"Pesan {ctx['n_items']} menu dari meja {ctx['table']}"]
            cands.append("Pesan lewat QR di meja" if ctx['source']=='QR' else "Pesan langsung ke kasir")
        else:
            cands+=["Pesan take away",f"Bungkus {ctx['i_full']} buat makan di rumah",f"Take away {ctx['when']} hari {ctx['dow']}"]
        if ctx['rush']: cands.append("Datang pas jam ramai")
        cands.append(f"Iseng coba {ctx['i_full']}")
        op=rng.choice(cands)
    if op: body=op+(". " if rng.random()<.5 else ", ")+(body if body[0].islower() else body[0].lower()+body[1:]) if op else body
    if op and not body.startswith(op): pass
    body=cap(body)
    # penutup
    if rng.random()<0.25 and pol_main in '+-':
        body+=" "+rng.choice(CLOSE_POS if pol_main=='+' else CLOSE_NEG)
    return finish(body,rng)
def finish(txt,rng):
    r=rng.random()
    if r<0.35: txt=formalize(txt)
    elif r<0.5: txt=very_casual(txt,rng)
    txt=re.sub(r'\s+',' ',txt).strip()
    txt=re.sub(r'\b(Tapi|Cuma|Untungnya|Sayangnya) ([A-Z])([a-z])',lambda m:m.group(1)+' '+m.group(2).lower()+m.group(3),txt)
    return txt

def compose(story,ctx,rng,short_ok=True,extra_gen=False):
    pol_main=story[0][1]
    if short_ok and rng.random()<0.06 and len(story)==1 and story[0][0] not in('kecepatan',):
        txt=rng.choice(SHORT_POS if pol_main=='+' else SHORT_NEG if pol_main=='-' else [rng.choice(B[('biasa','0')])])
        return finish(txt,rng)
    parts=[];pols=[]
    for asp,pol in story:
        parts.append(rng.choice(B[(asp,pol)]).format(i=ctx['i_main'],m=ctx['m'])); pols.append(pol)
    if extra_gen and rng.random()<.5:
        parts.append(rng.choice(GEN_POS if pol_main=='+' else GEN_NEG if pol_main=='-' else B[('biasa','0')])); pols.append(pol_main)
    body=cap(parts[0])
    for k in range(1,len(parts)):
        conn=rng.choice(OPEN_POS_NEG_CONN.get((pols[k-1],pols[k]),[", ",", "," dan "]))
        body+=conn+(cap(parts[k]) if conn.startswith('.') else parts[k])
    if not body.endswith('.'): body+='.'
    if rng.random()<0.5:
        c=[]
        if ctx['service']=='dine_in':
            c+=[f"Makan {ctx['when']} di sini hari {ctx['dow']}",f"Mampir {ctx['dow']} {ctx['when']}",f"Duduk di meja {ctx['table']}",f"Pesan {ctx['n_items']} menu dari meja {ctx['table']}",
                "Pesan lewat QR di meja" if ctx['source']=='QR' else "Pesan langsung ke kasir"]
        else:
            c+=["Pesan take away",f"Bungkus {ctx['i_full']} buat makan di rumah",f"Take away {ctx['when']} hari {ctx['dow']}"]
        if ctx['rush']: c.append("Datang pas jam ramai")
        c.append(f"Iseng coba {ctx['i_full']}")
        body=rng.choice(c)+". "+body
    if rng.random()<0.25 and pol_main in '+-':
        body+=" "+rng.choice(CLOSE_POS if pol_main=='+' else CLOSE_NEG)
    return finish(body,rng)

# ================= perluasan variasi (v3) =================
def _x(asp,pol,*xs): B[(asp,pol)].extend(xs)
_x('rasa','+',"{i} pas banget di lidah","{i} bumbunya berani dan seimbang","{i} wangi dan gurihnya terasa","dari semua yang dipesan, {i} paling juara","{i} lembut dan matang merata","{i} bikin lupa diet","{i} rasanya komplit","{i} gurih manisnya pas","{i} sesuai selera keluarga","{i} lebih enak dari dugaan saya","{i} rasanya bersih dan nggak bau","{i} bikin pengin nambah")
_x('rasa','-',"{i} terlalu asin menurut saya","{i} rasanya flat","{i} bumbunya kurang kerasa","{i} agak alot","{i} kematangannya nggak merata","{i} terlalu banyak minyak","{i} cuma enak di bagian awal","{i} rasanya kurang segar","{i} nggak seimbang rasanya","{i} bau kurang sedap","{i} tekstur kurang pas","{i} rasanya beda dari biasanya")
_x('porsi','+',"porsinya cukup untuk berdua","nasinya banyak","isian lauknya penuh","porsi mantap buat harga segini")
_x('porsi','-',"porsinya mepet","nasinya cuma sedikit","lauknya bikin kurang puas","porsi menyusut dibanding terakhir kali")
_x('kecepatan','+',"pesanan cepat sampai di meja","nggak lama langsung datang","cepat dan rapi penyajiannya","baru duduk sebentar pesanan sudah datang")
_x('kecepatan','-',"lama nunggu, kira-kira {m} menit","antreannya terasa lama, {m} menit baru keluar","nunggu terus sampai {m} menit","pesanan ketinggalan lama, hampir {m} menit")
_x('pelayanan','+',"pelayannya cekatan dan senyum","dilayani dengan baik dari awal","stafnya informatif soal menu","kasirnya teliti")
_x('pelayanan','-',"staf kurang perhatian","pesanan lupa diantar sampai ditanya","pelayannya terlihat kewalahan dan kurang ramah","tanya status pesanan malah nggak jelas jawabannya")
_x('harga','+',"harganya pas di kantong","sebanding dengan porsinya")
_x('harga','-',"harga terasa tinggi untuk rasa segitu","nggak sebanding dengan yang didapat")
_x('umum','+',"secara keseluruhan oke","senang makan di sini","kunjungan yang menyenangkan")
_x('umum','-',"secara keseluruhan nggak memuaskan","kunjungan kali ini kurang menyenangkan","tidak seperti yang saya harapkan")
_x('biasa','0',"ya standar warung makan pada umumnya","nggak buruk, nggak spesial","masih bisa diterima","cukup untuk mengisi perut")
CTX_OPEN=["Makan bareng keluarga","Ngumpul sama teman kantor","Makan siang sama rekan kerja","Mampir sehabis belanja","Lagi gajian jadi makan di sini","Makan berdua sama pasangan","Makan sendirian sepulang kerja","Ajak anak-anak makan di sini","Kumpul alumni di sini","Pertama kali mampir","Sudah beberapa kali ke sini","Ikut rekomendasi teman","Lagi pengin masakan rumahan","Datang setelah lihat menunya di QR","Mampir habis olahraga"]
CTX_CLOSE=["Semoga tetap konsisten.","Sampai ketemu lagi.","Itu pengalaman saya.","Segitu dulu ulasan dari saya.","Semoga bermanfaat buat yang mau mampir.","Terima kasih sudah melayani.","Itu aja sih.","Mungkin kapan-kapan coba menu lain."]
def compose(story,ctx,rng,short_ok=True,extra_gen=False):
    pol_main=story[0][1]
    if short_ok and rng.random()<0.07 and len(story)==1 and story[0][0]!='kecepatan':
        t=rng.choice(SHORT_POS if pol_main=='+' else SHORT_NEG if pol_main=='-' else [rng.choice(B[('biasa','0')])])
        if rng.random()<.5: t=f"{ctx['i_main']}: {t.lower()}" if pol_main!='0' else t
        return finish(t,rng)
    parts=[];pols=[]
    for asp,pol in story:
        parts.append(rng.choice(B[(asp,pol)]).format(i=ctx['i_main'],m=ctx['m'])); pols.append(pol)
    if extra_gen and rng.random()<.5:
        parts.append(rng.choice(GEN_POS if pol_main=='+' else GEN_NEG if pol_main=='-' else B[('biasa','0')])); pols.append(pol_main)
    body=cap(parts[0])
    for k in range(1,len(parts)):
        conn=rng.choice(OPEN_POS_NEG_CONN.get((pols[k-1],pols[k]),[", ",", "," dan "]))
        body+=conn+(cap(parts[k]) if conn.startswith('.') else parts[k])
    if not body.endswith('.'): body+='.'
    r=rng.random()
    if r<.62:
        c=[]
        if ctx['service']=='dine_in':
            c+=[f"Makan {ctx['when']} di sini hari {ctx['dow']}",f"Mampir {ctx['dow']} {ctx['when']}",f"Duduk di meja {ctx['table']}",f"Pesan {ctx['n_items']} menu dari meja {ctx['table']}",
                "Pesan lewat QR di meja" if ctx['source']=='QR' else "Pesan langsung ke kasir"]
        else:
            c+=["Pesan take away",f"Bungkus {ctx['i_full']} buat makan di rumah",f"Take away {ctx['when']} hari {ctx['dow']}"]
        if ctx['rush']: c+=["Datang pas jam ramai","Lagi ramai-ramainya pas saya datang"]
        c+=[f"Iseng coba {ctx['i_full']}",f"Pesan {ctx['i_full']}"]
        if ctx.get('i_other'): c+=[f"Pesan {ctx['i_full']} sama {ctx['i_other']}",f"Coba {ctx['i_other']} dan {ctx['i_full']}"]
        c+=CTX_OPEN*2
        body=rng.choice(c)+". "+body
    if rng.random()<0.3 and pol_main in '+-':
        body+=" "+rng.choice(CLOSE_POS if pol_main=='+' else CLOSE_NEG)
    elif rng.random()<0.15: body+=" "+rng.choice(CTX_CLOSE)
    return finish(body,rng)

## 2. Test set independen (100 komentar tulisan tangan)

In [ ]:
# Test set independen ditulis manual (bukan hasil template). (teks, label, kelompok)  kelompok: n=normal, h=sulit (negasi/kontras)
T=[]
def add(l,lab,g='n'):
    for t in l: T.append((t,lab,g))
add(["Ayam bakarnya empuk banget, bumbunya sampai ke dalam. Puas deh","Tempatnya adem, makanan datang cepat, kami sekeluarga senang","Es jeruknya segar, nasi gorengnya wangi. Recommended buat makan siang",
"Pelayan ramah banget, dikasih tambahan sambal tanpa diminta","Harga segini dapat porsi sebanyak ini, murah menurut saya","Gurame goreng crispy luar dalam, sambalnya juara","Sumpah enak, bakal balik lagi minggu depan",
"Makanan sampai masih panas dan rasanya konsisten enak","Soto-nya gurih, kuahnya bening tapi kaya rasa. Mantap","Dari awal datang disambut ramah, pesanan cepat selesai","Pesan lewat barcode gampang, nggak perlu antre. Praktis banget",
"Nila bakarnya matang merata dan wangi, nggak amis sama sekali","Kenyang dan puas, suasananya nyaman buat ngobrol lama","Mie gorengnya mantap, pedasnya pas, porsi lumayan besar","Overall pengalaman makan di sini menyenangkan",
"Jusnya seger, nggak terlalu manis. Cocok banget","Kwetiaunya juara, sayurnya masih renyah","Top lah pokoknya, nggak ada yang bikin kecewa","Tempat bersih, toiletnya juga terawat. Nilai plus","Sapo tahunya enak banget, hangat dan gurih",
"Lele gorengnya kriuk, nggak bau tanah. Suka","Paket keluarganya worth it, semua kenyang","Staf cekatan, meja langsung dibersihkan setelah kami selesai","Bumbu rawonnya pekat dan sedap, pasti pesan lagi","Datang rame-rame, semua pesanan keluar barengan dan enak",
"Es teh jumbonya mantul, pas buat siang panas","Pas dibungkus, kemasannya rapi dan kuah nggak tumpah","Murah meriah tapi rasanya nggak murahan","Pesanan lengkap, rasanya oke, pelayanan memuaskan","Salah satu tempat makan favorit saya di sini, konsisten tiap datang",
"Saya suka sambal matahnya, pedas segar","Makan malam yang menyenangkan, terima kasih ya","Enak, enak, enak! Nambah nasi dua kali","Cepat dan enak, nggak pakai nunggu lama","Ikan bakar di sini the best, wangi arang",
"Bagus banget pelayanannya, sopan dan sigap","Nasi gorengnya medok, telurnya pas matang","Menu banyak pilihan dan semuanya terjamin enak","Rasanya otentik, mirip masakan ibu di rumah","Pokoknya puas dari awal sampai bayar"],'positive')
add(["Nunggu hampir satu jam, makanan baru keluar. Kecewa berat","Nasinya dingin dan ayamnya alot, rasanya mengecewakan","Pelayannya cuek, ditanya malah diam saja","Porsi kecil banget buat harga segitu, nggak worth it","Ikannya amis dan bumbunya hambar, nggak akan pesan lagi",
"Pesanan saya salah dan nggak ada yang minta maaf","Meja kotor, banyak lalat, bikin nggak selera makan","Sudah bayar lewat QRIS tapi pesanan tak kunjung diproses","Sup-nya keasinan parah, nggak bisa dimakan","Lama banget, teman sampai selesai makan baru punyaku datang",
"Mie gorengnya berminyak dan hambar, zonk","Minuman datang paling akhir dan esnya sudah cair semua","Harga mahal tapi rasa biasa saja, sayang uangnya","Ayam goreng terlalu kering, rasanya aneh","Kapok, nggak balik lagi",
"Tempatnya pengap dan panas, makanan juga lama","Kuah tumpah ke mana-mana pas dibungkus, kemasan asal-asalan","Dipanggil berkali-kali nggak ada yang datang","Rasanya jauh dari ekspektasi, sangat mengecewakan","Nasi goreng gosong di bagian bawah, pahit",
"Staf judes, bikin nggak nyaman","Aplikasi pemesanan error terus, akhirnya harus ke kasir","Pesan dari tadi belum juga datang, parah","Lele gorengnya bau tanah, nyesel pesan","Porsinya pelit, lauk cuma sepotong kecil",
"Bumbu kurang, rasa datar banget","Sudah antre lama, ternyata menu yang dipesan habis, nggak dikabari","Makanan sampai udah dingin, nggak enak lagi","Pengalaman terburuk, nggak recommended sama sekali","Sambalnya pedas kebangetan sampai perut sakit",
"Piring dan sendok masih kotor, jijik","Overpriced, rasa nggak sebanding","Nunggu 45 menit untuk sepiring nasi? nggak masuk akal","Es tehnya cuma manis gula, nggak ada rasa tehnya","Kecewa, ayamnya masih berdarah di tengah",
"Kasirnya ketus dan salah hitung","Salah banget pilih makan di sini","Rasa tidak karuan, bikin eneg","Sayurnya layu dan nggak segar","Lambat banget, bikin kesel"],'negative')
add(["Nggak mengecewakan sama sekali, malah di luar dugaan enak","Tidak ada yang kurang, semuanya pas","Nggak nyesel datang ke sini","Pelayanannya tidak lambat, malah cepat","Rasanya nggak hambar, bumbunya terasa","Tidak mahal untuk kualitas segini","Dikira bakal lama, ternyata cepat banget","Awalnya ragu, tapi ternyata enak banget",
"Memang agak ramai, tapi tetap dilayani dengan baik dan makanannya enak","Porsi nggak kecil, cukup bikin kenyang"],'positive','h')
add(["Enak sih, tapi nunggunya lama banget sampai nggak sabar, nggak akan balik","Tempatnya bagus, sayang makanannya hambar dan mahal","Ayamnya enak tapi pelayanannya parah, males datang lagi","Bukan makanan yang enak, jauh dari kata layak","Nggak ada yang istimewa, rasanya mengecewakan",
"Katanya recommended, ternyata biasa dan kemahalan","Lumayan cepat, tapi rasanya nggak enak sama sekali","Tidak puas, tidak akan kembali","Suasana oke, tapi minumannya nggak segar dan nasinya dingin, kecewa","Awalnya senang sama tempatnya, akhirnya kecewa sama rasanya"],'negative','h')

## 3. Definisi 7 metode, korpus latih sintetis, fungsi evaluasi

In [ ]:
import sys,random,re,json,pickle,numpy as np,collections,warnings
warnings.filterwarnings('ignore')
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline,make_union
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold,cross_val_predict
from sklearn.metrics import f1_score,precision_score,recall_score,brier_score_loss,accuracy_score
# ---------- korpus latih sintetis (tidak pernah memuat kalimat test set) ----------
EXTRA_POS=["mantap sekali rasanya","pelayanannya bagus dan cepat","saya sangat puas","makanannya lezat dan segar","suasananya nyaman","enak dan harganya terjangkau","pasti datang lagi","porsi pas dan bumbu mantap","stafnya baik","sangat recommended","tempatnya bersih","dimasak dengan baik dan hangat"]
EXTRA_NEG=["rasanya tidak enak","pelayanan lambat dan tidak ramah","saya sangat kecewa","makanan dingin dan hambar","tempatnya kotor","mahal tapi tidak enak","tidak akan datang lagi","porsi kecil dan bumbu kurang","stafnya kasar","sangat tidak recommended","terlalu lama menunggu","masakan gosong dan asin"]
NEGW=[("tidak ","tidak "),]
def mk(n,seed=0):
    r=random.Random(seed); X=[];y=[]
    keys_p=[k for k in B if k[1]=='+']; keys_n=[k for k in B if k[1]=='-']
    pos_bank=[p for k in keys_p for p in B[k]]+GEN_POS+EXTRA_POS
    neg_bank=[p for k in keys_n for p in B[k]]+GEN_NEG+EXTRA_NEG
    def fill(s): return s.format(i=r.choice(['makanannya','nasi gorengnya','ayamnya','minumannya','menunya','sotonya','ikannya']),m=r.choice([10,20,30,40,45,60]))
    for _ in range(n):
        lab=r.random()<.5
        main=fill(r.choice(pos_bank if lab else neg_bank)); k=r.random()
        if k<.55: t=main
        elif k<.8:   # klausa kedua searah
            t=main+r.choice([", ",", dan ",". "])+fill(r.choice(pos_bank if lab else neg_bank))
        else:        # kontras: label mengikuti klausa terakhir (mayoritas penulis menaruh kesan akhir di belakang)
            t=fill(r.choice(neg_bank if lab else pos_bank))+r.choice([", tapi ",". Tapi ",", sayangnya ",", cuma "][:2] if lab else [", tapi ",". Tapi "])+main
        if r.random()<.35: t=formalize(t)
        elif r.random()<.2: t=very_casual(t,r)
        X.append(t.lower() if r.random()<.5 else t); y.append(int(lab))
    return X,np.array(y)
Xtr,ytr=mk(4000,1)
Xte=[t for t,_,_ in T]; yte=np.array([1 if l=='positive' else 0 for _,l,_ in T]); gte=np.array([g for *_,g in T])
# --------- leksikon baseline ---------
POSW=set("enak mantap lezat gurih segar ramah puas bagus cepat sigap nyaman bersih murah suka juara recommended worth terjangkau empuk wangi sedap top kenyang hangat renyah praktis memuaskan menyenangkan senang otentik pas konsisten the best nikmat".split())
NEGW_=set("hambar amis alot gosong asin keasinan dingin lama lambat kotor mahal kecewa mengecewakan parah zonk kapok cuek judes ketus pelit kecil berminyak pahit eneg error salah jijik nyesel buruk terburuk kemahalan overpriced".split())
NEG_FLIP={'nggak','tidak','bukan','gak','ga','tak'}
def lex(t,neg=True):
    w=re.findall(r"[a-z]+",t.lower()); s=0
    for i,x in enumerate(w):
        p=1 if x in POSW else -1 if x in NEGW_ else 0
        if p and neg and i>0 and w[i-1] in NEG_FLIP: p=-p
        s+=p
    return s
from sklearn.base import BaseEstimator,ClassifierMixin
class Lex(ClassifierMixin,BaseEstimator):
    def __init__(self,neg=True): self.neg=neg
    def fit(s,X,y):
        s.classes_=np.array([0,1]); return s
    def predict_proba(s,X):
        sc=np.array([lex(t,s.neg) for t in X],float); p=1/(1+np.exp(-1.2*sc)); return np.c_[1-p,p]
    def predict(s,X): return (s.predict_proba(X)[:,1]>.5).astype(int)
def wc(): return make_union(TfidfVectorizer(ngram_range=(1,2),min_df=2,sublinear_tf=True,lowercase=True),TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),min_df=3,sublinear_tf=True))
MODELS={
 'M1 Leksikon (aturan kata, tanpa negasi)':lambda:Lex(False),
 'M1b Leksikon + penanganan negasi':lambda:Lex(True),
 'M2 TF-IDF kata + Naive Bayes':lambda:make_pipeline(TfidfVectorizer(ngram_range=(1,2),min_df=2,sublinear_tf=True),MultinomialNB(alpha=.3)),
 'M3 TF-IDF kata+char + Regresi Logistik':lambda:make_pipeline(wc(),LogisticRegression(C=4,max_iter=2000,class_weight='balanced')),
 'M4 TF-IDF kata+char + LinearSVM (terkalibrasi)':lambda:make_pipeline(wc(),CalibratedClassifierCV(LinearSVC(C=.5,class_weight='balanced'),cv=5,method='sigmoid')),
}
def ece(p,y,b=10):
    conf=np.maximum(p,1-p); pred=(p>.5).astype(int); acc=(pred==y).astype(float); e=0
    for i in range(b):
        m=(conf>i/b*.5+.5)&(conf<=(i+1)/b*.5+.5)
        if m.sum(): e+=m.mean()*abs(acc[m].mean()-conf[m].mean())
    return e
def boot(y,pred,B_=2000,seed=0):
    r=np.random.RandomState(seed);v=[]
    for _ in range(B_):
        i=r.randint(0,len(y),len(y)); v.append(f1_score(y[i],pred[i],average='macro'))
    return np.percentile(v,[2.5,97.5])
from sklearn.preprocessing import FunctionTransformer
import scipy.sparse as sp
def negtag(t):
    w=re.findall(r"[a-z]+",t.lower()); o=[];flip=0
    for x in w:
        if x in NEG_FLIP: flip=3; o.append('NEG'); continue
        o.append(('n_'+x) if flip>0 else x); flip=max(0,flip-1)
    return ' '.join(o)
def negcol(X): return [negtag(t) for t in X]
def lexfeat(X): return sp.csr_matrix(np.array([[lex(t,True),lex(t,False)] for t in X],float)/3)
def wcn(): return make_union(TfidfVectorizer(ngram_range=(1,2),min_df=2,sublinear_tf=True),TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),min_df=3,sublinear_tf=True))
MODELS['M5 TF-IDF + penanda negasi + Regresi Logistik']=lambda:make_pipeline(FunctionTransformer(negcol),TfidfVectorizer(ngram_range=(1,2),min_df=2,sublinear_tf=True),LogisticRegression(C=4,max_iter=2000,class_weight='balanced'))
MODELS['M6 Hibrida: leksikon-negasi + TF-IDF(negasi) + Regresi Logistik']=lambda:make_pipeline(make_union(make_pipeline(FunctionTransformer(negcol),TfidfVectorizer(ngram_range=(1,2),min_df=2,sublinear_tf=True)),FunctionTransformer(lexfeat)),LogisticRegression(C=4,max_iter=2000,class_weight='balanced'))

## 4. Muat `reviews_v2.csv`

In [ ]:
import os,pandas as pd
REV=pd.read_csv('reviews_v2.csv').to_dict('records') if os.path.exists('reviews_v2.csv') else []
print('jumlah review dimuat:',len(REV))

## 5. Evaluasi: 7 metode × skema latih A/B + cek in-domain

In [ ]:
# ---- bagian ini mengasumsikan: MODELS, mk, Xtr, ytr, Xte, yte, gte, ece, boot, REV (list dict: comment, rating) sudah ada ----
import numpy as np,warnings; warnings.filterwarnings('ignore')
from sklearn.metrics import f1_score,accuracy_score,precision_score,recall_score,brier_score_loss
from sklearn.model_selection import StratifiedKFold
Xrv=[r['comment'] for r in REV if int(r['rating'])!=3]; yrv=np.array([1 if int(r['rating'])>=4 else 0 for r in REV if int(r['rating'])!=3])
print('review berlabel (tanpa rating 3):',len(Xrv),'positif',int(yrv.sum()),'negatif',int((1-yrv).sum()))
REGIMES={'A: latih sintetis saja':(Xtr,ytr)}
if len(Xrv): REGIMES['B: sintetis + %d review berlabel rating'%len(Xrv)]=(list(Xtr)+Xrv,np.r_[ytr,yrv])
RES={}
for rg,(Xa,ya) in REGIMES.items():
    for nm,f in MODELS.items():
        m=f().fit(Xa,ya); p=m.predict_proba(Xte)[:,1]; pr=(p>.5).astype(int); hh=gte=='h'
        lo,hi=boot(yte,pr)
        RES[(rg,nm)]=dict(acc=accuracy_score(yte,pr),macro_f1=f1_score(yte,pr,average='macro'),ci=(lo,hi),f1_n=f1_score(yte[~hh],pr[~hh],average='macro'),f1_h=f1_score(yte[hh],pr[hh],average='macro'),
            neg_prec=precision_score(yte,pr,pos_label=0),neg_rec=recall_score(yte,pr,pos_label=0),brier=brier_score_loss(yte,p),ece=ece(p,yte),p=p)
        r_=RES[(rg,nm)]; print(f"[{rg[:1]}] {nm[:44]:44s} F1={r_['macro_f1']:.3f} [{lo:.3f},{hi:.3f}] sulit={r_['f1_h']:.3f} negP={r_['neg_prec']:.2f} negR={r_['neg_rec']:.2f} Brier={r_['brier']:.3f} ECE={r_['ece']:.3f}")
# cek in-domain: 5-fold CV hanya pada review (optimistis, karena komentar berasal dari template yang sama)
skf=StratifiedKFold(5,shuffle=True,random_state=0); INDOM={}
if not len(Xrv): print('reviews_v2.csv tidak ada: skema B dan cek in-domain dilewati')
Xr_a=np.array(Xrv,dtype=object)
for nm,f in (MODELS.items() if len(Xrv) else []):
    oof=np.zeros(len(yrv),int)
    for tr,te_ in skf.split(Xr_a,yrv):
        mm=f().fit(list(Xr_a[tr]),yrv[tr]); oof[te_]=(mm.predict_proba(list(Xr_a[te_]))[:,1]>.5).astype(int)
    INDOM[nm]=f1_score(yrv,oof,average='macro'); print(f"in-domain 5-fold {nm[:44]:44s} F1={INDOM[nm]:.3f}")

## 6. Tabel ringkas

In [ ]:
import pandas as pd
rows=[dict(skema=rg[:1],metode=nm[:60],F1=round(v['macro_f1'],3),CI=f"{v['ci'][0]:.3f}-{v['ci'][1]:.3f}",F1_sulit=round(v['f1_h'],3),Brier=round(v['brier'],3),ECE=round(v['ece'],3),in_domain=round(INDOM.get(nm,float('nan')),3)) for (rg,nm),v in RES.items()]
pd.DataFrame(rows).sort_values(['skema','F1'],ascending=[True,False])

## 7. Model final M6 + prediksi out-of-fold untuk 2.000 review
Tiap review diprediksi model yang tidak pernah melihatnya (5-fold). Skor = keyakinan pada label terprediksi.

In [ ]:
from sklearn.model_selection import StratifiedKFold
m6=[k for k in MODELS if k.startswith("M6")][0]
def conf_label(p): return [("positive",round(float(x),2)) if x>=.5 else ("negative",round(float(1-x),2)) for x in p]
if REV:
    df=pd.DataFrame(REV); rt=df["rating"].astype(int).values; txt=df["comment"].astype(str).values
    lab=np.where(rt>=4,1,np.where(rt<=2,0,-1)); P=np.zeros(len(df))
    for tr,te_ in StratifiedKFold(5,shuffle=True,random_state=42).split(txt,rt):
        trl=[i for i in tr if lab[i]>=0]
        mm=MODELS[m6]().fit(list(Xtr)+[txt[i] for i in trl],np.r_[ytr,lab[trl]]); P[te_]=mm.predict_proba(list(txt[te_]))[:,1]
    out=conf_label(P); df["sentiment_model"]=[a for a,_ in out]; df["score_model"]=[b for _,b in out]
    m=lab>=0; print("kesesuaian dengan rating (tanpa rating 3):",round(float(((P[m]>=.5)==(lab[m]==1)).mean()),3))
    print(df["sentiment_model"].value_counts().to_dict())
    df.to_csv("reviews_v2_prediksi.csv",index=False)
    final=MODELS[m6]().fit(list(Xtr)+list(txt[m]),np.r_[ytr,lab[m]])
else:
    final=MODELS[m6]().fit(Xtr,ytr)
print(conf_label(final.predict_proba(["Enak banget, pelayanannya ramah","Nunggu sejam, makanan dingin","nggak enak sama sekali","Nggak mengecewakan sama sekali"])[:,1]))